In [26]:
# TODO
# Make notebook runable
# Update paths
# Update input data
# Delete IFFs and nurse estimates

In [27]:
import pandas as pd
import numpy as np
import pickle

from tjn_tools.style_guide import *
from tjn_tools.data_processing import *
import tjn_tools
from config_new import YEAR_CBCR, YEAR_SOTJ, UNILATERAL_CROSS, UNILATERAL_PANEL, BILATERAL_CROSS

Note: Originally this was named 98.combine_parts.ipynb, but I renamed it to 98.combine_parts.ipynb to make it run last.

In [28]:
# Defines file paths for different directories related to estimations data
path_files = "../../data/raw/estimations/"
path_files_final = "../../data/final/estimations/"
path_files_final_analysis = "../../data/final/analysis/"
path_files_temp = "../../data/intermediate/estimations/"
path_figures = "../../data/final/estimations/figures/"

In [29]:
# TODO Transfer these paths to config file
# Define file path for offshore wealth tax evasion data
tax_evasion_file_output = f"{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Data/Projects/2006 Offshore wealth/Tables/Offshore wealth Table Full results raw.xlsx"

In [30]:
# Define info expenditures file path and columns of interest to be used in the analysis
info_expenditures_output = f"{path_files_final}{YEAR_CBCR}_info_expenditures_new.csv"
cols_other_info = ['who_gvt_health_expenditure','govt_exp_educ_gdp',
        'total_taxes_revenue', 'cit_revenue', 'iso3', 'gdp', 'population',
        'region_tjn',"ukt","oecd","oecd_oct","g20","eu28","month_wage","fsi_2022_rank","fsi_2022_score","cthi_2021_rank","cthi_2021_share","cthi_2021_score"]


In [31]:
# Read cbcr etr rates data
etr_output = f"{path_files_final}{YEAR_CBCR}_cbcr_etr_rates_new.xlsx"
df_etrs = pd.read_excel(etr_output,index_col=0)
# Create dictionary with iso3 as key and etr as value
iso3_to_etr = df_etrs["ETR_total"].to_dict()

In [32]:
# Define tax avoidance sotj tavle file path to be used in the analysis
file_output = f"{YEAR_CBCR}_tax_avoidance_sotj_table_new.xlsx"
sotj_table_output = f"{path_files_final_analysis}{file_output}"

In [33]:
# Read series iso3 to corporate income tax (cit) dictionnary
iso3_to_cit_output = f"{path_files_final}{YEAR_CBCR}_iso3_to_cit_new.dump"
iso3_to_cit = pickle.load(open(iso3_to_cit_output,"rb+"))

In [34]:
# Define file paths for combined output. One for saving the output locally and another in SharePoint
combined_table = f"{YEAR_CBCR}combined_output.xlsx"
workstream_path = f"{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Scale of Tax Injustice/State of Tax Justice report/{YEAR_SOTJ} Report/Combined_offshore_corporate/"
final_table_output = f"{path_files_final_analysis}{combined_table}"
final_table_output_workstream = f"{workstream_path}{combined_table}"

# 1. Read data

In [35]:
# Read cthi unilateral cross data for 2021 - TODO Update with latest data
# Note:Some variables were not added before: "EU28 OECT", "EU27", "EU27 OCT", "GBR OCT", th_eu_blacklist_201006, th_eu_greylist_201006, th_unctad2015
countryYearBase = pd.read_stata(f"{tjn_tools.paths.sharepoint_root}/TJN - Shared Documents/Workstreams/Financial Secrecy/CTHI/CTHI-2021/GSW/210211 countryYearBase for CTHI2021.dta")
# Select columns of interest
countryYearBase = countryYearBase.loc[:,["country","EU27","EU27_OCT","EU28_OCT","GBR_OCT","th_eu_blacklist_201006", "th_eu_greylist_201006", "th_unctad2015"]]
# Generate iso3 column
countryYearBase["iso3"] = countryYearBase["country"].apply(get_iso3, print_failure = False)
# Clean dataframe
countryYearBase = countryYearBase.loc[countryYearBase["country"] != "West Bank and Gaza"]
countryYearBase = countryYearBase.drop(columns=["country"])
countryYearBase = countryYearBase.drop_duplicates()
countryYearBase.sample(10)

,EU27,EU27_OCT,EU28_OCT,GBR_OCT,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015,iso3
12388,0.0,0.0,0.0,0.0,0.0,0.0,1.0,NIU
1824,0.0,0.0,1.0,1.0,0.0,0.0,1.0,BMU
5016,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ECU
17482,0.0,0.0,0.0,0.0,0.0,1.0,0.0,TUR
13072,0.0,0.0,0.0,0.0,0.0,0.0,NaN,PSE
12236,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NER
4408,0.0,1.0,1.0,0.0,0.0,0.0,0.0,CUW
18394,0.0,0.0,0.0,0.0,0.0,0.0,0.0,URY
18470,0.0,0.0,0.0,0.0,0.0,0.0,0.0,UZB
2432,0.0,0.0,1.0,1.0,0.0,0.0,0.0,IOT


In [36]:
# Read info expenditures data and merge it with countryYearBase
other_info = pd.read_csv(info_expenditures_output, sep="\t", usecols=cols_other_info)
other_info = pd.merge(other_info,countryYearBase,how="left")
other_info.head()

,iso3,total_taxes_revenue,cit_revenue,govt_exp_educ_gdp,population,month_wage,who_gvt_health_expenditure,gdp,cthi_2021_rank,cthi_2021_share,cthi_2021_score,fsi_2022_rank,fsi_2022_score,region_tjn,eu28,oecd,g20,ukt,oecd_oct,EU27,EU27_OCT,EU28_OCT,GBR_OCT,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015
0,ABW,6.458101e+08,NaN,NaN,105962.0,1986.169974,NaN,3.202235e+09,56.0,0.002126,70.134565,75.0,70.925,Caribbean/American isl.,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
1,AFG,1.689714e+09,NaN,5.893645e+08,36686784.0,82.518273,8.951096e+07,1.841885e+10,NaN,NaN,NaN,NaN,NaN,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,AGO,1.514723e+10,NaN,1.590632e+09,31273533.0,291.116457,8.314383e+08,7.779294e+10,NaN,NaN,NaN,33.0,79.450,Africa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,AIA,NaN,NaN,NaN,NaN,NaN,NaN,2.930103e+08,39.0,0.005760,100.000000,58.0,75.450,Caribbean/American isl.,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
4,ALA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Europe,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
# Read Unilateral cross data with selected columns
cols = {"iso3":"iso3","ps_jansky19": "TA: JP 2019 (USD million)"} #Update this list for all useful comparisons (TWZ, C&J)
# ext_estimates = pd.read_csv(UNILATERAL_CROSS,skiprows=1,sep="\t",usecols=(list(cols.keys())))
ext_estimates = pd.read_csv(UNILATERAL_CROSS,usecols=(list(cols.keys())))
# cols = [_ for _ in ohter_m.columns if ("ps_" in _) and not ("_so") in _ and (_ not in ("ps_cobham2018","ps_torslov2018") )]
# Filter by dropñing rows with least two nan values and remove duplicates
ext_estimates = ext_estimates[list(cols)].dropna(thresh=2).drop_duplicates(subset=["iso3"])
# Rename columns
ext_estimates = ext_estimates.rename(columns=cols)
# Convert values from USD million to USD except for the first column
ext_estimates[list(ext_estimates.columns )[1:]] /= 1E6

In [38]:
# Merge other info with ext_estimates
other_info = pd.merge(other_info,ext_estimates,how="left",validate="1:1")
other_info.head()

,iso3,total_taxes_revenue,cit_revenue,govt_exp_educ_gdp,population,month_wage,who_gvt_health_expenditure,gdp,cthi_2021_rank,cthi_2021_share,cthi_2021_score,fsi_2022_rank,fsi_2022_score,region_tjn,eu28,oecd,g20,ukt,oecd_oct,EU27,EU27_OCT,EU28_OCT,GBR_OCT,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015,TA: JP 2019 (USD million)
0,ABW,6.458101e+08,NaN,NaN,105962.0,1986.169974,NaN,3.202235e+09,56.0,0.002126,70.134565,75.0,70.925,Caribbean/American isl.,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,NaN
1,AFG,1.689714e+09,NaN,5.893645e+08,36686784.0,82.518273,8.951096e+07,1.841885e+10,NaN,NaN,NaN,NaN,NaN,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
2,AGO,1.514723e+10,NaN,1.590632e+09,31273533.0,291.116457,8.314383e+08,7.779294e+10,NaN,NaN,NaN,33.0,79.450,Africa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN
3,AIA,NaN,NaN,NaN,NaN,NaN,NaN,2.930103e+08,39.0,0.005760,100.000000,58.0,75.450,Caribbean/American isl.,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,NaN
4,ALA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Europe,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
#Income class
# Read Unilateral panel data with selected columns
income_class = pd.read_csv(UNILATERAL_PANEL,usecols=["iso3","year","gdp","population"])
# Calculate GDP per capita
income_class["GDPpc"] = income_class["gdp"]/income_class["population"]
# Calculate income class
income_class["income_class"] = pd.cut(income_class["GDPpc"],[0,1025,3995,12375,np.inf],labels=["L","LM","UM","H"]) #These thresholds are for 2018 (from analytical classifications here: http://databank.worldbank.org/data/download/site-content/OGHIST.xls)
# Identify the ISO3 codes for which the "IncomeClass" information is missing by calculating the set difference between all ISO3 codes and the ISO3 codes with available "IncomeClass" information
missing_income = set(income_class["iso3"]) - set(income_class.dropna(subset=["income_class"])["iso3"])
print(missing_income)
# Separate the rows from the "income_class" DataFrame into two subsets based on the presence or absence of "IncomeClass" information, dropping any duplicate rows and renaming columns in one of the subsets
info_on_income_class = income_class.loc[~income_class["iso3"].isin(missing_income)].dropna(subset=["income_class"]).drop_duplicates(subset=["iso3"],keep="last")
no_info_on_income_class = income_class.loc[income_class["iso3"].isin(missing_income)].dropna(subset=["income_class"]).drop_duplicates(subset=["iso3"],keep="last")
no_info_on_income_class = no_info_on_income_class.rename(columns = {"income_class": "income_class_na","income_class": "income_class"})
# Combine the "info_on_income_class" and "no_info_on_income_class" DataFrames into a single DataFrame named "income_class" and replaces the values in the "IncomeClass" column with descriptive labels.
income_class = pd.concat([info_on_income_class, no_info_on_income_class])
income_class["income_class"] = income_class["income_class"].map({'H':"High income", 'L':"Low income", 'LM':"Lower-middle income", 'UM':"Upper-middle income"})
# Keep columns of interest
income_class = income_class.loc[:,["iso3","income_class"]]
# Merge other info with income_class
other_info = pd.merge(other_info,income_class,how="left",validate="1:1")
other_info.head()

{'IOT', 'UMI', 'FLK', 'CXR', 'TKL', 'BVT', 'ALA', 'MYT', 'PCN', 'PAL', 'MSR', 'ATF', 'NFK', 'BLM', 'SGS', 'SHN', 'GUF', 'NIU', 'WLF', 'JEY', 'GGY', 'GLP', 'COK', 'XXK', 'HMD', 'VAT', 'BES', 'TMP', 'WSH', 'CCK', 'SJM', 'TWN', 'SPM', 'REU', 'AIA', 'ANT', 'ESH', 'MTQ'}


,iso3,total_taxes_revenue,cit_revenue,govt_exp_educ_gdp,population,month_wage,who_gvt_health_expenditure,gdp,cthi_2021_rank,cthi_2021_share,cthi_2021_score,fsi_2022_rank,fsi_2022_score,region_tjn,eu28,oecd,g20,ukt,oecd_oct,EU27,EU27_OCT,EU28_OCT,GBR_OCT,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015,TA: JP 2019 (USD million),income_class
0,ABW,6.458101e+08,NaN,NaN,105962.0,1986.169974,NaN,3.202235e+09,56.0,0.002126,70.134565,75.0,70.925,Caribbean/American isl.,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,NaN,High income
1,AFG,1.689714e+09,NaN,5.893645e+08,36686784.0,82.518273,8.951096e+07,1.841885e+10,NaN,NaN,NaN,NaN,NaN,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,Low income
2,AGO,1.514723e+10,NaN,1.590632e+09,31273533.0,291.116457,8.314383e+08,7.779294e+10,NaN,NaN,NaN,33.0,79.450,Africa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,Lower-middle income
3,AIA,NaN,NaN,NaN,NaN,NaN,NaN,2.930103e+08,39.0,0.005760,100.000000,58.0,75.450,Caribbean/American isl.,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN
4,ALA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Europe,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
#Tax avoidance
# Read an Excel file into the "tax_avoidance_file" DataFrame, adjusts the values in the "CIT" and "ETR" columns by dividing them by 100, maps country names to ISO-3 codes, and replaces a specific code in the "ISO-3 code of country" column
tax_avoidance_file = pd.read_excel(sotj_table_output).drop_duplicates()
tax_avoidance_file["CIT"] /= 100
tax_avoidance_file["ETR"] /= 100
tax_avoidance_file["Name"].loc[tax_avoidance_file["Name"] == "Cura�ao"] = "Curacao"
tax_avoidance_file["ISO-3 code of country"] = tax_avoidance_file["Name"].map(get_iso3).replace("SCG","SRB") 

tax_avoidance_file.head()

 Africa not matched to any file
 Asia not matched to any file
 Caribbean/American isl. not matched to any file
 Europe not matched to any file
 Latin America not matched to any file
 Northern America not matched to any file
 Oceania not matched to any file


,Name,MNCs,CIT,ETR,Profit loss (M),Min. Profit loss (M),Max. Profit loss (M),Revenue loss using CIT (M),Min. Revenue loss using CIT (M),Max. Revenue loss using CIT (M),Revenue loss using ETR (M),Min Revenue loss using ETR (M),Max. Revenue loss using ETR (M),Robust,3+reporters,N_reporters,gdp,pop,Profit loss per gdp (%),Profit loss per health (%),Profit loss per educ (%),Profit loss per tax_revenue (%),Profit loss per pop ($ per capita),ISO-3 code of country
0,Africa,Foreign (in)Foreign (out)Foreign (in)Foreign (...,0.2787,0.2917,17290,14833,19893,5035.8,4350.8,5753.6,4525.6,3970.5,5102.6,121,56,745,5241974361408,2524099480,0.3,22.6,5.7,1.9,7,NaN
1,Algeria,Foreign (in),0.2600,0.5117,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1,1,14,174910878623,41927007,0.0,0.0,0.0,0.0,0,DZA
2,Algeria,Foreign (out),0.2600,0.5117,32,29,35,8.3,7.6,9.1,16.3,14.9,17.9,2,1,14,174910878623,41927007,0.0,0.4,0.3,0.0,1,DZA
3,Angola,Foreign (in),0.3000,0.2884,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,1,1,8,77792944472,31273533,0.0,0.0,0.0,0.0,0,AGO
4,Angola,Foreign (out),0.3000,0.2884,424,361,487,127.1,108.3,146.1,122.2,104.1,140.5,2,1,8,77792944472,31273533,0.5,51.0,26.6,2.8,14,AGO


In [41]:
#Tax evasion
# The code reads an Excel file into the "tax_evasion_file" DataFrame, maps country names to ISO-3 codes, removes the "Country" column, and displays the initial rows of the updated DataFrame.
tax_evasion_file = pd.read_excel(tax_evasion_file_output)
tax_evasion_file["ISO-3 code of country"] = tax_evasion_file["Country"].map(get_iso3)
tax_evasion_file = tax_evasion_file.drop(columns=["Country"])
tax_evasion_file.head()


,Share of global offshore wealth owned by citizens of country,Offshore wealth owned by citizens of country (USD billion),Offshore wealth owned by citizens of country (% of GDP),Tax revenue loss: Offshore wealth (USD million),Share of global tax loss inflicted by country,Tax loss inflicted on other countries,ISO-3 code of country
0,3.197117e-09,0.000032,0.000118,0.000495,0.0,0.0,PLW
1,2.780527e-08,0.000276,0.000044,0.004302,0.0,0.0,GUM
2,1.538021e-07,0.001529,0.000085,0.021331,0.0,0.0,PRK
3,1.454730e-07,0.001446,0.012234,0.016934,0.0,0.0,NRU
4,7.445466e-07,0.007402,0.024528,0.000000,0.0,0.0,AIA


In [42]:
#other_info.loc[other_info["iso3"].isin([get_iso3(_) for _ in ["Malaysia","Vietnam","Thailand","Cambodia","Indonesia","Myanmar","Philippines"]]),["iso3","month_wage"]]

In [43]:
# Select specific columns from the "tax_avoidance_file" DataFrame where the "MNCs" column is equal to "Domestic", "Foreign (out)", or "Foreign (in)", and assigns the resulting DataFrame to "ta_dom", "ta_fo", and "ta_ga" respectively. It then renames the columns in each DataFrame to include corresponding suffixes indicating the type of MNCs
ta_dom = tax_avoidance_file.loc[tax_avoidance_file["MNCs"]=="Domestic",["ISO-3 code of country","Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)',"N_reporters"]]   
ta_dom.columns = list(ta_dom.columns[:1]) + [_ + " - dom" for _ in ta_dom.columns[1:]]
ta_fo = tax_avoidance_file.loc[tax_avoidance_file["MNCs"]=="Foreign (out)",["ISO-3 code of country","ETR","Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)',"Revenue loss using CIT (M)","Revenue loss using ETR (M)","Robust","N_reporters"]]
ta_fo.columns = list(ta_fo.columns[:2]) + [_ + " - for lose" for _ in ta_fo.columns[2:]]
ta_ga = tax_avoidance_file.loc[tax_avoidance_file["MNCs"]=="Foreign (in)",["ISO-3 code of country","CIT","Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)',"Revenue loss using CIT (M)","Revenue loss using ETR (M)","Robust"]]
ta_ga.columns = list(ta_ga.columns[:2]) + [_ + " - for gain" for _ in ta_ga.columns[2:]]

# Concatenate the "ta_dom", "ta_fo", and "ta_ga" DataFrames along the column axis, with the ISO-3 code of country as the common index, and assigns the result to the "ta" DataFrame
ta = pd.concat([ta_dom.set_index("ISO-3 code of country"),ta_fo.set_index("ISO-3 code of country"),ta_ga.set_index("ISO-3 code of country")],axis=1,sort=False)
ta = ta.reset_index().rename(columns={"index":"ISO-3 code of country"})

# Calculate revenue loss values based on profit loss and tax rates for different types of MNCs and assigns them to the "ta" DataFrame
for tax in ["CIT","ETR"]:
    for var in ["Profit loss (M)",'Min. Profit loss (M)', 'Max. Profit loss (M)']:
        for end in ["- dom","- for lose","- for gain"]:
            ta[f"Revenue loss using {tax} (M) {end}"] = ta[f"{var} {end}"]*ta[tax]

ta = ta.rename(columns={"N_reporters - dom":"Reporting country"})
ta.head()

,ISO-3 code of country,Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,Reporting country,ETR,Profit loss (M) - for lose,Min. Profit loss (M) - for lose,Max. Profit loss (M) - for lose,Revenue loss using CIT (M) - for lose,Revenue loss using ETR (M) - for lose,Robust - for lose,N_reporters - for lose,CIT,Profit loss (M) - for gain,Min. Profit loss (M) - for gain,Max. Profit loss (M) - for gain,Revenue loss using CIT (M) - for gain,Revenue loss using ETR (M) - for gain,Robust - for gain,Revenue loss using CIT (M) - dom,Revenue loss using ETR (M) - dom
0,ZAF,844.0,788.0,901.0,1.0,0.1795,4011.0,3850.0,4230.0,1184.4000,759.2850,2.0,22.0,0.2800,0.0,0.0,0.0,0.00,0.000,1.0,252.2800,161.7295
1,HKG,-19850.0,-19872.0,-19825.0,1.0,0.0645,1359.0,1301.0,1410.0,232.6500,90.9450,2.0,25.0,0.1650,-36033.0,-37185.0,-34512.0,-5694.48,-2226.024,2.0,-3271.1250,-1278.7125
2,IND,33781.0,33541.0,34087.0,1.0,0.4132,29565.0,28204.0,31243.0,15096.6176,12909.6076,2.0,22.0,0.4832,0.0,0.0,0.0,0.00,0.000,1.0,16470.8384,14084.7484
3,IDN,-2200.0,-2240.0,-2161.0,1.0,0.2701,10592.0,10134.0,11343.0,2835.7500,3063.7443,2.0,20.0,0.2500,0.0,0.0,0.0,0.00,0.000,1.0,-540.2500,-583.6861
4,JPN,11681.0,10960.0,12560.0,1.0,0.2815,510.0,487.0,525.0,156.1350,147.7875,2.0,21.0,0.2974,0.0,0.0,0.0,0.00,0.000,1.0,3735.3440,3535.6400


In [44]:
def return_gain(common_var = "Profit loss (M) - ",vars_=["dom","for lose","for gain"],sign=-1):
    """ Calculate the sum of averaged values from the "ta" DataFrame based on the given parameters, printing intermediate results, and returns the resulting values"""
    vals = ((ta["{}{}".format(common_var,vars_[0])]+sign*ta["{}{}".format(common_var,vars_[0])].abs())/2).fillna(0)
    print(vals)
    for v in vars_[1:]:
        vals += ((ta["{}{}".format(common_var,v)]+sign*ta["{}{}".format(common_var,v)].abs())/2).fillna(0)
    print(vals)
    return vals

# Iterate over variations "dom", "for lose", and "for gain", and for each variation, it iterates over states "" (empty string), "Min. ", and "Max. ". It fills missing values in columns of the "ta" DataFrame with names in the format "{st}Profit loss (M) - {v}" (where "{st}" represents the state and "{v}" represents the variation) with 0.
for v in ["dom","for lose","for gain"]:
    for st in ["","Min. ","Max. "]:
        ta[f"{st}Profit loss (M) - {v}"] = ta[f"{st}Profit loss (M) - {v}"].fillna(0)
# Maps the values in the "ISO-3 code of country" column of the "ta" DataFrame to corresponding values from the "iso3_to_etr" and "iso3_to_cit" dictionaries, and assigns the mapped values to the "ETR" and "CIT" columns of the "ta" DataFrame, respectively.
ta["ETR"] = ta["ISO-3 code of country"].map(iso3_to_etr)
ta["CIT"] = ta["ISO-3 code of country"].map(iso3_to_cit)
# Calculate and assigns the summed profit gain and profit loss values based on different variations and states to the corresponding columns in the "ta" DataFrame
for st in ["","Min. ","Max. "]:
    ta[f"{st}Profit gain (M)"] = -return_gain(common_var = f"{st}Profit loss (M) - ",vars_=["dom","for lose","for gain"],sign=-1)
    ta[f"{st}Profit loss (M)"] = return_gain(common_var = f"{st}Profit loss (M) - ",vars_=["dom","for lose","for gain"],sign=1)
# Calculate and assigns the revenue gain and revenue loss values based on different variations and states using the "CIT" and "ETR" columns in the "ta" DataFrame multiplied by the corresponding profit gain and profit loss values, and assigns the calculated values to the corresponding columns in the "ta" DataFrame
for st in ["","Min. ","Max. "]:
    ta[f"{st}Revenue gain using CIT (M)"] = ta["CIT"]*ta[f"{st}Profit gain (M)"]
    ta[f"{st}Revenue gain using ETR (M)"] = ta["ETR"]*ta[f"{st}Profit gain (M)"]
    ta[f"{st}Revenue loss using CIT (M)"] = ta["CIT"]*ta[f"{st}Profit loss (M)"]
    ta[f"{st}Revenue loss using ETR (M)"] = ta["ETR"]*ta[f"{st}Profit loss (M)"]

# Assign the sum of the "Reporting country" column and a binary indicator (1 if "N_reporters - for lose" is greater than 3, 0 otherwise) to the "Robust" column in the "ta" DataFrame
ta["Robust"] = ta["Reporting country"]+((ta["N_reporters - for lose"])>3).astype(int)

ta.sort_values(by="Profit gain (M)").tail(20)

0          0.0
1     -19850.0
2          0.0
3      -2200.0
4          0.0
        ...   
208        0.0
209        0.0
210        0.0
211        0.0
212        0.0
Name: Profit loss (M) - dom, Length: 213, dtype: float64
0          0.0
1     -55883.0
2          0.0
3      -2200.0
4          0.0
        ...   
208        0.0
209        0.0
210        0.0
211     -787.0
212        0.0
Name: Profit loss (M) - dom, Length: 213, dtype: float64
0        844.0
1          0.0
2      33781.0
3          0.0
4      11681.0
        ...   
208        0.0
209        0.0
210        0.0
211        0.0
212        0.0
Name: Profit loss (M) - dom, Length: 213, dtype: float64
0       4855.0
1       1359.0
2      63346.0
3      10592.0
4      12191.0
        ...   
208        0.0
209        0.0
210        0.0
211        0.0
212        0.0
Name: Profit loss (M) - dom, Length: 213, dtype: float64
0          0.0
1     -19872.0
2          0.0
3      -2240.0
4          0.0
        ...   
208        0.0
209    

,ISO-3 code of country,Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,Reporting country,ETR,Profit loss (M) - for lose,Min. Profit loss (M) - for lose,Max. Profit loss (M) - for lose,Revenue loss using CIT (M) - for lose,Revenue loss using ETR (M) - for lose,Robust - for lose,N_reporters - for lose,CIT,Profit loss (M) - for gain,Min. Profit loss (M) - for gain,Max. Profit loss (M) - for gain,Revenue loss using CIT (M) - for gain,Revenue loss using ETR (M) - for gain,Robust - for gain,Revenue loss using CIT (M) - dom,Revenue loss using ETR (M) - dom,Profit gain (M),Profit loss (M),Min. Profit gain (M),Min. Profit loss (M),Max. Profit gain (M),Max. Profit loss (M),Revenue gain using CIT (M),Revenue gain using ETR (M),Revenue loss using CIT (M),Revenue loss using ETR (M),Min. Revenue gain using CIT (M),Min. Revenue gain using ETR (M),Min. Revenue loss using CIT (M),Min. Revenue loss using ETR (M),Max. Revenue gain using CIT (M),Max. Revenue gain using ETR (M),Max. Revenue loss using CIT (M),Max. Revenue loss using ETR (M),Robust
14,DNK,-8447.0,-8487.0,-8404.0,1.0,0.156665,2342.0,2234.0,2504.0,550.8800,392.3768,2.0,17.0,0.22000,0.0,0.0,0.0,0.0000,0.0000,1.0,-1848.8800,-1316.9068,8447.0,2342.0,8487.0,2234.0,8404.0,2504.0,1858.339990,1323.350754,515.239997,366.909846,1867.139990,1329.617361,491.479997,349.990006,1848.879990,1316.614151,550.879997,392.289604,2.0
126,BRB,0.0,0.0,0.0,NaN,0.012663,245.0,223.0,273.0,68.2500,3.4671,2.0,11.0,0.25000,-10606.0,-11890.0,-9714.0,-2428.5000,-123.3678,2.0,NaN,NaN,10606.0,245.0,11890.0,223.0,9714.0,273.0,2651.500000,134.307449,61.250000,3.102520,2972.500000,150.567186,55.750000,2.823926,2428.500000,123.011744,68.250000,3.457093,NaN
21,IMN,-394.0,-395.0,-394.0,1.0,0.076711,234.0,220.0,256.0,0.0000,19.6352,2.0,9.0,0.00000,-12885.0,-14015.0,-12177.0,-0.0000,-933.9759,2.0,-0.0000,-30.2198,13279.0,234.0,14410.0,220.0,12571.0,256.0,0.000000,1018.643545,0.000000,17.950342,0.000000,1105.403531,0.000000,16.876390,0.000000,964.332255,0.000000,19.637981,2.0
5,MYS,-16634.0,-16702.0,-16567.0,1.0,0.218637,3193.0,3034.0,3406.0,817.4400,744.5516,2.0,21.0,0.24000,0.0,0.0,0.0,0.0000,0.0000,1.0,-3976.0800,-3621.5462,16634.0,3193.0,16702.0,3034.0,16567.0,3406.0,3992.159911,3636.807297,766.319983,698.107833,4008.479910,3651.674610,728.159984,663.344556,3976.079911,3622.158620,817.439982,744.677507,2.0
27,NOR,-18237.0,-18273.0,-18199.0,1.0,0.332999,4136.0,3870.0,4444.0,1022.1200,1479.8520,2.0,15.0,0.23000,0.0,0.0,0.0,0.0000,0.0000,1.0,-4185.7700,-6060.2670,18237.0,4136.0,18273.0,3870.0,18199.0,4444.0,4194.510076,6072.905895,951.280017,1377.284574,4202.790076,6084.893866,890.100016,1288.706795,4185.770076,6060.251927,1022.120019,1479.848319,2.0
127,VGB,0.0,0.0,0.0,NaN,0.004546,1354.0,1287.0,1491.0,0.0000,6.7095,2.0,22.0,0.00000,-21087.0,-23164.0,-20106.0,-0.0000,-90.4770,2.0,NaN,NaN,21087.0,1354.0,23164.0,1287.0,20106.0,1491.0,0.000000,95.852893,0.000000,6.154731,0.000000,105.294087,0.000000,5.850177,0.000000,91.393667,0.000000,6.777477,NaN
38,MEX,-21496.0,-21626.0,-21368.0,1.0,0.200267,19455.0,19022.0,19871.0,5961.3000,3980.1613,2.0,22.0,0.30000,0.0,0.0,0.0,0.0000,0.0000,1.0,-6410.4000,-4280.0104,21496.0,19455.0,21626.0,19022.0,21368.0,19871.0,6448.800256,4304.935453,5836.500232,3896.190884,6487.800258,4330.970139,5706.600227,3809.475353,6410.400255,4279.301301,5961.300237,3979.501879,2.0
133,PRI,0.0,0.0,0.0,NaN,0.027705,470.0,443.0,506.0,197.3400,14.0162,2.0,13.0,0.39000,-23003.0,-24696.0,-21741.0,-8478.9900,-602.2257,2.0,NaN,NaN,23003.0,470.0,24696.0,443.0,21741.0,506.0,8971.170000,637.297785,183.300000,13.021343,9631.440000,684.202326,172.770000,12.273309,8478.990000,602.334093,197.340000,14.018723,NaN
8,KOR,-29827.0,-30041.0,-29573.0,1.0,0.300256,964.0,928.0,1006.0,276.6500,302.1018,2.0,21.0,0.27500,0.0,0.0,0.0,0.0000,0.0000,1.0,-8132.5750,-8880.7719,29827.0,964.0,30041.0,928.0,29573.0,1006.0,8202.425178,8955.730050,265.100006,289.446601,8261.275179,9019.984794,255.200006,278.637392,81

In [45]:
# Select the rows in the "ta" DataFrame where the "ISO-3 code of country" is equal to "USA" and retrieves the columns containing "Revenue loss" and "CIT" in their names (excluding columns with decimal points) for those rows
ta.loc[ta["ISO-3 code of country"]=="USA",[_ for _ in ta.columns if ("Revenue loss" in _) and ("." not in _) and ("CIT" in _)]]

,Revenue loss using CIT (M) - for lose,Revenue loss using CIT (M) - for gain,Revenue loss using CIT (M) - dom,Revenue loss using CIT (M)
42,11844.36,0.0,126874.08,137577.96


In [46]:
# Select the rows in the "ta" DataFrame where the "ISO-3 code of country" is equal to "TCD" and retrieves the columns containing "Revenue loss" and "CIT" in their names (excluding columns with decimal points) for those rows
ta.loc[ta["ISO-3 code of country"]=="TCD",[_ for _ in ta.columns if ("Revenue loss" in _) and ("." not in _) and ("CIT" in _)]]

,Revenue loss using CIT (M) - for lose,Revenue loss using CIT (M) - for gain,Revenue loss using CIT (M) - dom,Revenue loss using CIT (M)
52,0.7,-13.65,NaN,0.35


In [47]:
# Select the rows in the "ta" DataFrame where the "ISO-3 code of country" is equal to "TCD" and retrieves the columns containing "Profit loss" in their names (excluding columns with decimal points) for those rows.
ta.loc[ta["ISO-3 code of country"]=="TCD",[_ for _ in ta.columns if ("Profit loss" in _) and ("." not in _) ]]

,Profit loss (M) - dom,Profit loss (M) - for lose,Profit loss (M) - for gain,Profit loss (M)
52,0.0,1.0,-49.0,1.0


In [48]:
## MERGE FILES
# Merge the "tax_evasion_file" DataFrame with the "ta" DataFrame using an outer join, and then merges the resulting DataFrame with the "other_info" DataFrame based on the "ISO-3 code of country" column, using a left join. The resulting DataFrame is assigned to the variable "df_merged"
df_merged = pd.merge(tax_evasion_file,ta,how="outer").dropna(subset=["ISO-3 code of country"])
# df_merged = pd.merge(df_merged,iff_file,how="outer").dropna(subset=["ISO-3 code of country"]) #TODO delete this line and nurses-wise one
# df_merged = pd.merge(df_merged,childrens_file,how="left").dropna(subset=["ISO-3 code of country"])
df_merged = pd.merge(df_merged,other_info,left_on="ISO-3 code of country",right_on="iso3",how="left")

df_merged["Offshore wealth owned by citizens of country (USD billion)"] *= 1000*0.05 #Convert to million and multiply by the rate of return
#df_merged["fsi_2022_share"] *= 100
#df_merged["cthi_2021_share"] *= 100

# Map the "CIT" values in the "df_merged" DataFrame based on the "ISO-3 code of country" using the "iso3_to_cit" dictionary.
df_merged["CIT"] = df_merged["ISO-3 code of country"].map(iso3_to_cit)
df_merged.head()


,Share of global offshore wealth owned by citizens of country,Offshore wealth owned by citizens of country (USD billion),Offshore wealth owned by citizens of country (% of GDP),Tax revenue loss: Offshore wealth (USD million),Share of global tax loss inflicted by country,Tax loss inflicted on other countries,ISO-3 code of country,Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,Reporting country,ETR,Profit loss (M) - for lose,Min. Profit loss (M) - for lose,Max. Profit loss (M) - for lose,Revenue loss using CIT (M) - for lose,Revenue loss using ETR (M) - for lose,Robust - for lose,N_reporters - for lose,CIT,Profit loss (M) - for gain,Min. Profit loss (M) - for gain,Max. Profit loss (M) - for gain,Revenue loss using CIT (M) - for gain,Revenue loss using ETR (M) - for gain,Robust - for gain,Revenue loss using CIT (M) - dom,Revenue loss using ETR (M) - dom,Profit gain (M),Profit loss (M),Min. Profit gain (M),Min. Profit loss (M),Max. Profit gain (M),Max. Profit loss (M),Revenue gain using CIT (M),Revenue gain using ETR (M),Revenue loss using CIT (M),Revenue loss using ETR (M),Min. Revenue gain using CIT (M),Min. Revenue gain using ETR (M),Min. Revenue loss using CIT (M),Min. Revenue loss using ETR (M),Max. Revenue gain using CIT (M),Max. Revenue gain using ETR (M),Max. Revenue loss using CIT (M),Max. Revenue loss using ETR (M),Robust,iso3,total_taxes_revenue,cit_revenue,govt_exp_educ_gdp,population,month_wage,who_gvt_health_expenditure,gdp,cthi_2021_rank,cthi_2021_share,cthi_2021_score,fsi_2022_rank,fsi_2022_score,region_tjn,eu28,oecd,g20,ukt,oecd_oct,EU27,EU27_OCT,EU28_OCT,GBR_OCT,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015,TA: JP 2019 (USD million),income_class
0,3.197117e-09,0.001589,0.000118,0.000495,0.0,0.0,PLW,0.0,0.0,0.0,NaN,0.000000,2.0,2.0,2.0,NaN,0.00,1.0,1.0,0.000,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,-0.0,2.0,-0.0,2.0,-0.0,2.0,-0.00,-0.000000,0.0,0.00000,-0.00,-0.000000,0.00,0.000000,-0.00,-0.000000,0.00,0.000000,NaN,PLW,6.044141e+07,0.0,1.860397e+07,17864.0,1181.432026,1.823826e+07,2.849000e+08,NaN,NaN,NaN,NaN,NaN,Oceania,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,Upper-middle income
1,2.780527e-08,0.013822,0.000044,0.004302,0.0,0.0,GUM,0.0,0.0,0.0,NaN,0.050050,90.0,81.0,99.0,20.79,4.95,1.0,4.0,0.210,-2.0,-2.0,-2.0,-0.42,-0.1,1.0,NaN,NaN,2.0,90.0,2.0,81.0,2.0,99.0,0.42,0.100099,18.9,4.50446,0.42,0.100099,17.01,4.054014,0.42,0.100099,20.79,4.954905,NaN,GUM,NaN,NaN,NaN,168678.0,2284.404018,NaN,6.056000e+09,NaN,NaN,NaN,134.0,70.300,Oceania,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,High income
2,1.538021e-07,0.076457,0.000085,0.021331,0.0,0.0,PRK,0.0,0.0,0.0,NaN,0.222528,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.325,0.0,0.0,0.0,0.00,NaN,0.0,NaN,NaN,-0.0,0.0,-0.0,0.0,-0.0,0.0,-0.00,-0.000000,0.0,0.00000,-0.00,-0.000000,0.00,0.000000,-0.00,-0.000000,0.00,0.000000,NaN,PRK,NaN,NaN,NaN,25638149.0,104.751097,NaN,1.748726e+10,NaN,NaN,NaN,NaN,NaN,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,Low income
3,1.454730e-07,0.072316,0.012234,0.016934,0.0,0.0,NRU,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NRU,3.674134e+07,NaN,7.288936e+06,11924.0,840.401398,9.831266e+06,1.240214e+08,NaN,NaN,NaN,139.0,59.075,Oceania,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,NaN,Upper-middle income
4,7.445466e-07,0.370123,0.024528,0.000000,0.0,0.0,AIA,0.0,0.0,0.0,NaN,0.000000,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.000,0.0,0.0,0.0,0.00,NaN,0.0,NaN,NaN,-0.0,0.0,-0.0,0.0,-0.0,0.0,-0.00,-0.000000,0.0,0.00000,-0.00,-0.000000,0.00,0.000000,-0.00,-0.000000,0.00,0.000000,NaN,AIA,NaN,NaN,NaN,NaN,NaN,NaN,2.930103e+08,39.0,0.00576,100.0,58.0,75.450,Caribbean/American isl.,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN


In [49]:
tax_avoidance_file.loc[tax_avoidance_file["ISO-3 code of country"]=="TCD"]

,Name,MNCs,CIT,ETR,Profit loss (M),Min. Profit loss (M),Max. Profit loss (M),Revenue loss using CIT (M),Min. Revenue loss using CIT (M),Max. Revenue loss using CIT (M),Revenue loss using ETR (M),Min Revenue loss using ETR (M),Max. Revenue loss using ETR (M),Robust,3+reporters,N_reporters,gdp,pop,Profit loss per gdp (%),Profit loss per health (%),Profit loss per educ (%),Profit loss per tax_revenue (%),Profit loss per pop ($ per capita),ISO-3 code of country
15,Chad,Foreign (in),0.35,0.0,-49,-63,-39,-17.2,-22.1,-13.7,0.0,0.0,0.0,1,0,2,11239167048,15604210,-0.4,-62.9,-19.4,-6.1,-3,TCD
16,Chad,Foreign (out),0.35,0.0,1,1,2,0.5,0.4,0.6,0.0,0.0,0.0,1,0,2,11239167048,15604210,0.0,1.8,0.6,0.2,0,TCD


In [50]:
# Create a dictionary called "rename_cols" and assigns new column names to the respective keys based on the original column names in the DataFrame. The new column names are formatted to represent specific tax-related metrics in USD million.
rename_cols = {}
for st in ["","Min. ","Max. "]:
    rename_cols[f"{st}Profit gain (M)"] = f"TA: {st}Tax base gain (USD million)"
    rename_cols[f"{st}Profit loss (M)"] = f"TA: {st}Tax base loss (USD million)"
    rename_cols[f"{st}Revenue gain using CIT (M)"] = f"TA: {st}Tax revenue gain using CIT (USD million)"
    rename_cols[f"{st}Revenue gain using ETR (M)"] = f"TA: {st}Tax revenue gain using ETR (USD million)"
    rename_cols[f"{st}Revenue loss using CIT (M)"] = f"TA: {st}Tax revenue loss using CIT (USD million)"
    rename_cols[f"{st}Revenue loss using ETR (M)"] = f"TA: {st}Tax revenue loss using ETR (USD million)"
    
# Rename columns
rename_cols.update({"Offshore wealth owned by citizens of country (USD billion)": 'OW: Tax base loss (USD million)',
 "Tax revenue loss: Offshore wealth (USD million)": 'OW: Tax revenue loss (USD million)',
 'Share of global tax loss inflicted by country': 'Harm OW: Total (% total)',
 'Tax loss inflicted on other countries': 'Harm OW: Total (USD million)',
 'Reporting country': 'TA: Reporting country',
 'Robust': 'TA: Robust',
 'N_reporters - for lose': 'TA: Number countries reporting',
 'Outward Banking Positions': 'IFF: Outward Banking Positions',
 'Inward Banking Positions': 'IFF: Inward Banking Positions',
 'Outward FDI': 'IFF: Outward FDI',
 'Inward FDI': 'IFF: Inward FDI',
 'Outward Portfolio Inv.': 'IFF: Outward Portfolio Inv.',
 'Inward Portfolio Inv.': 'IFF: Inward Portfolio Inv.',
 'Outward Trade (Exports)': 'IFF: Outward Trade (Exports)',
 'Inward Trade (Imports)': 'IFF: Inward Trade (Imports)',
 'Top Flow': 'IFF: Top Flow',
 'Top Vulnerability': 'IFF: Top Vulnerability',
 'Vulnerability Region': 'IFF: Vulnerability Region',
 'Top1': 'IFF: Top1',
 'Top2': 'IFF: Top2',
 'Top3': 'IFF: Top3',
 'fsi_2022_rank': 'FSI_Rank',
 #'FSI2022_Share': 'FSI_Share',
 'fsi_2022_score': 'FSI_Score',
 'cthi_2021_rank': 'CTHI_Rank',
 'cthi_2021_share': 'CTHI_Share',
 'cthi_2021_score': 'CTHI_Score',
 #'Children lives lost due to tax revenue loss (total)': 'Children lives lost due to tax revenue loss (total)',
 'govt_exp_educ_gdp': 'WBD: Government education expenditure',
 'who_gvt_health_expenditure': 'WHO: Government health expenditure',
 'total_taxes_revenue': 'GRD: Total tax revenue',
 'cit_revenue': 'GRD: Total corporate income revenue',
 'gdp': 'GDP',
 'population': 'POP',
 'region_tjn': 'Region',
 'income_class': 'Income Class',
 'ukt': 'UK territory',
 'month_wage': 'Average wage'})

In [51]:
# Rename columns, drops rows with missing values in specific columns, excludes a particular ISO-3 code, creates a "Country" column using ISO-3 code mapping, sets a specific region for an ISO-3 code, creates an "IncomeClass2" column based on "Income Class", and displays the resulting DataFrame "df_merged"
df_merged = df_merged.rename(columns = rename_cols)
df_merged = df_merged.dropna(subset=["OW: Tax base loss (USD million)","Harm OW: Total (USD million)","TA: Tax base loss (USD million)","TA: Tax base gain (USD million)"],how="all")
df_merged = df_merged.loc[df_merged["ISO-3 code of country"]!="ATA"]
df_merged["Country"] = df_merged["ISO-3 code of country"].map(iso3_to_name)
df_merged.loc[df_merged["ISO-3 code of country"]=="PUS","Region"] = 'Caribean/American isl.'
df_merged["Income Class"] = df_merged["Income Class"].str.contains("Low").replace({False: "Higher", True: "Lower"})
df_merged.head()

,Share of global offshore wealth owned by citizens of country,OW: Tax base loss (USD million),Offshore wealth owned by citizens of country (% of GDP),OW: Tax revenue loss (USD million),Harm OW: Total (% total),Harm OW: Total (USD million),ISO-3 code of country,Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,TA: Reporting country,ETR,Profit loss (M) - for lose,Min. Profit loss (M) - for lose,Max. Profit loss (M) - for lose,Revenue loss using CIT (M) - for lose,Revenue loss using ETR (M) - for lose,Robust - for lose,TA: Number countries reporting,CIT,Profit loss (M) - for gain,Min. Profit loss (M) - for gain,Max. Profit loss (M) - for gain,Revenue loss using CIT (M) - for gain,Revenue loss using ETR (M) - for gain,Robust - for gain,Revenue loss using CIT (M) - dom,Revenue loss using ETR (M) - dom,TA: Tax base gain (USD million),TA: Tax base loss (USD million),TA: Min. Tax base gain (USD million),TA: Min. Tax base loss (USD million),TA: Max. Tax base gain (USD million),TA: Max. Tax base loss (USD million),TA: Tax revenue gain using CIT (USD million),TA: Tax revenue gain using ETR (USD million),TA: Tax revenue loss using CIT (USD million),TA: Tax revenue loss using ETR (USD million),TA: Min. Tax revenue gain using CIT (USD million),TA: Min. Tax revenue gain using ETR (USD million),TA: Min. Tax revenue loss using CIT (USD million),TA: Min. Tax revenue loss using ETR (USD million),TA: Max. Tax revenue gain using CIT (USD million),TA: Max. Tax revenue gain using ETR (USD million),TA: Max. Tax revenue loss using CIT (USD million),TA: Max. Tax revenue loss using ETR (USD million),TA: Robust,iso3,GRD: Total tax revenue,GRD: Total corporate income revenue,WBD: Government education expenditure,POP,Average wage,WHO: Government health expenditure,GDP,CTHI_Rank,CTHI_Share,CTHI_Score,FSI_Rank,FSI_Score,Region,eu28,oecd,g20,UK territory,oecd_oct,EU27,EU27_OCT,EU28_OCT,GBR_OCT,th_eu_blacklist_201006,th_eu_greylist_201006,th_unctad2015,TA: JP 2019 (USD million),Income Class,Country
0,3.197117e-09,0.001589,0.000118,0.000495,0.0,0.0,PLW,0.0,0.0,0.0,NaN,0.000000,2.0,2.0,2.0,NaN,0.00,1.0,1.0,0.000,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,-0.0,2.0,-0.0,2.0,-0.0,2.0,-0.00,-0.000000,0.0,0.00000,-0.00,-0.000000,0.00,0.000000,-0.00,-0.000000,0.00,0.000000,NaN,PLW,6.044141e+07,0.0,1.860397e+07,17864.0,1181.432026,1.823826e+07,2.849000e+08,NaN,NaN,NaN,NaN,NaN,Oceania,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,Higher,Palau
1,2.780527e-08,0.013822,0.000044,0.004302,0.0,0.0,GUM,0.0,0.0,0.0,NaN,0.050050,90.0,81.0,99.0,20.79,4.95,1.0,4.0,0.210,-2.0,-2.0,-2.0,-0.42,-0.1,1.0,NaN,NaN,2.0,90.0,2.0,81.0,2.0,99.0,0.42,0.100099,18.9,4.50446,0.42,0.100099,17.01,4.054014,0.42,0.100099,20.79,4.954905,NaN,GUM,NaN,NaN,NaN,168678.0,2284.404018,NaN,6.056000e+09,NaN,NaN,NaN,134.0,70.300,Oceania,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,Higher,Guam
2,1.538021e-07,0.076457,0.000085,0.021331,0.0,0.0,PRK,0.0,0.0,0.0,NaN,0.222528,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.325,0.0,0.0,0.0,0.00,NaN,0.0,NaN,NaN,-0.0,0.0,-0.0,0.0,-0.0,0.0,-0.00,-0.000000,0.0,0.00000,-0.00,-0.000000,0.00,0.000000,-0.00,-0.000000,0.00,0.000000,NaN,PRK,NaN,NaN,NaN,25638149.0,104.751097,NaN,1.748726e+10,NaN,NaN,NaN,NaN,NaN,Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,Lower,North Korea
3,1.454730e-07,0.072316,0.012234,0.016934,0.0,0.0,NRU,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NRU,3.674134e+07,NaN,7.288936e+06,11924.0,840.401398,9.831266e+06,1.240214e+08,NaN,NaN,NaN,139.0,59.075,Oceania,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,NaN,Higher,Nauru
4,7.445466e-07,0.370123,0.024528,0.000000,0.0,0.0,AIA,0.0,0.0,0.0,NaN,0.000000,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.000,0.0,0.0,0.0,0.00,NaN,0.0,NaN,NaN,-0.0,0.0,-0.0,0.0,-0.0,0.0,-0.00,-0.000000,0.0,0.00000,-0.00,-0.000000,0.00,0.000000,-0.00,-0.000000,0.00,0.000000,NaN,AIA,NaN,NaN,NaN,NaN,NaN,NaN,2.930103e+0

In [52]:
#Impute missing values
# Calculate health expenditure as a percentage of GDP, education expenditure as a percentage of GDP, total tax revenue as a percentage of GDP, and corporate income revenue as a percentage of GDP, and assigns these values to the corresponding columns in the DataFrame "df_merged"
df_merged["health_gdp"] = df_merged["WHO: Government health expenditure"]/df_merged["GDP"]
df_merged["educ_gdp"] = df_merged["WBD: Government education expenditure"]/df_merged["GDP"]
df_merged["tax_rev_gdp"] = df_merged["GRD: Total tax revenue"]/df_merged["GDP"]
df_merged["c_tax_rev_gdp"] = df_merged["GRD: Total corporate income revenue"]/df_merged["GDP"]

# Calculate the average values of government health expenditure as a percentage of GDP, government education expenditure as a percentage of GDP, total tax revenue as a percentage of GDP, and corporate income revenue as a percentage of GDP, grouped by income class, and assigns these values to the corresponding columns in the DataFrame "reg_av"
reg_av = df_merged.groupby("Income Class").sum()
reg_av["health_gdp"] = reg_av["WHO: Government health expenditure"]/reg_av["GDP"]
reg_av["educ_gdp"] = reg_av["WBD: Government education expenditure"]/reg_av["GDP"]
reg_av["tax_rev_gdp"] = reg_av["GRD: Total tax revenue"]/reg_av["GDP"]
reg_av["c_tax_rev_gdp"] = reg_av["GRD: Total corporate income revenue"]/reg_av["GDP"]

# Convert it to dictionnary
reg_av = reg_av.to_dict()

# Assign the average values of the columns "health_gdp", "educ_gdp", "tax_rev_gdp", and "c_tax_rev_gdp" from the DataFrame "reg_av" to the corresponding NaN values in the same columns of the DataFrame "df_merged" based on the income class.
for v in ["health_gdp","educ_gdp","tax_rev_gdp","c_tax_rev_gdp"]:
    df_merged.loc[np.isnan(df_merged[v]),v] =  df_merged.loc[np.isnan(df_merged[v]),"Income Class"].map(reg_av[v])

# Impute the values for government health expenditure, government education expenditure, total tax revenue, and total corporate income revenue in "df_merged" by multiplying the GDP values with the corresponding ratios.
df_merged["WHO: Government health expenditure (imp)"] = df_merged["health_gdp"]*df_merged["GDP"]
df_merged["WBD: Government education expenditure (imp)"] = df_merged["educ_gdp"]*df_merged["GDP"]
df_merged["GRD: Total tax revenue (imp)"] = df_merged["tax_rev_gdp"]*df_merged["GDP"]
df_merged["GRD: Total corporate income revenue (imp)"] = df_merged["c_tax_rev_gdp"]*df_merged["GDP"]

In [53]:
#Tax losses
# Update the "df_merged" dataframe by adding columns for tax revenue losses using CIT and ETR, while also handling missing values and replacing negative values with 0 in the "Loss OW" column
df_merged["Loss OW (USD million)"] = df_merged['OW: Tax revenue loss (USD million)'].fillna(0)
df_merged.loc[df_merged["Loss OW (USD million)"]<0,"Loss OW (USD million)"] = 0
for st in ["","Min. ","Max. "]:
    df_merged[f"{st}Loss TA using CIT (USD million)"] = df_merged['TA: Tax revenue loss using CIT (USD million)'].fillna(0)
    df_merged[f"{st}Loss TA using ETR (USD million)"] = df_merged['TA: Tax revenue loss using ETR (USD million)'].fillna(0)

# Create a new column "GDP (only lossers)" in the "df_merged" dataframe and sets its values to the same as the "GDP" column, replacing any negative values with 0.
df_merged["GDP (only lossers)"] = df_merged["GDP"].copy()
df_merged.loc[df_merged["GDP (only lossers)"]<0,"GDP (only lossers)"] = 0

# Calculate various loss metrics for each tax type (CIT and ETR) in the "df_merged" dataframe, including total loss in USD million, percentage of GDP, percentage of government tax revenue, global and regional percentages of GDP and government tax revenue, per capita loss, percentages relative to government education and health expenditures, government corporate income revenue, and the number of nurses
for tax in ["CIT","ETR"]:
    df_merged[f"Loss Total using {tax} (USD million)"] = df_merged[f"Loss TA using {tax} (USD million)"].fillna(0) + df_merged["Loss OW (USD million)"].fillna(0)
    df_merged[f"Loss Total using {tax} (% GDP)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GDP"]
    df_merged[f"Loss Total using {tax} (% gvt tax revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GRD: Total tax revenue"]
    df_merged[f"Loss Total using {tax} Global (% GDP)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"].sum()/df_merged["GDP"].sum()
    df_merged[f"Loss Total using {tax} Regional (% GDP)"] = 100*1E6*df_merged.groupby("Region")[f"Loss Total using {tax} (USD million)"].transform(sum)/df_merged.groupby("Region")["GDP"].transform(sum)
    df_merged[f"Loss Total using {tax} Global (% gvt tax revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"].sum()/df_merged["GRD: Total tax revenue (imp)"].sum()
    df_merged[f"Loss Total using {tax} Regional (% gvt tax revenue)"] = 100*1E6*df_merged.groupby("Region")[f"Loss Total using {tax} (USD million)"].transform(sum)/df_merged.groupby("Region")["GRD: Total tax revenue (imp)"].transform(sum)
    df_merged[f"Loss Total using {tax} (per capita)"] = 1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["POP"]
    df_merged[f"Loss Total using {tax} (% Education)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged['WBD: Government education expenditure']
    df_merged[f"Loss Total using {tax} (% Health)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["WHO: Government health expenditure"]
    df_merged[f"Loss Total using {tax} (%  gvt corporate revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GRD: Total corporate income revenue"]
    df_merged[f"Loss Total using {tax} (%  gvt tax revenue)"] = 100*1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["GRD: Total tax revenue"]
    df_merged[f"Loss Total using {tax} (# Nurses)"] = 1E6*df_merged[f"Loss Total using {tax} (USD million)"]/df_merged["Average wage"]/12


In [54]:
# Retrieve specific loss metrics related to CIT (Corporate Income Tax) in the USA, including the loss total as a percentage of government tax revenue, the loss total in USD million, the loss in tax revenue using CIT in USD million, and the loss in offshore wealth in USD million. These metrics help assess the impact of tax-related losses in the country.
df_merged.loc[df_merged["ISO-3 code of country"]=="USA",[f"Loss Total using CIT (% gvt tax revenue)","Loss Total using CIT (USD million)","Loss TA using CIT (USD million)","Loss OW (USD million)"]]

,Loss Total using CIT (% gvt tax revenue),Loss Total using CIT (USD million),Loss TA using CIT (USD million),Loss OW (USD million)
216,4.674673,176091.057656,137577.96,38513.097656


In [55]:
df_merged["TA: Tax revenue loss using ETR (USD million)"].sum()
print("TRL {0:2,.0f}B (95% CI {1:2,.0f}-{2:2,.0f}B)".format(*1e-3*df_merged[["TA: Tax revenue loss using CIT (USD million)","TA: Min. Tax revenue loss using CIT (USD million)","TA: Max. Tax revenue loss using CIT (USD million)"]].sum()))

TRL 300B (95% CI 294-307B)


In [56]:
#Harm to others
#OW: Already in the file
df_merged["Harm OW: Total (% total)"] *= 100

# Calculate various columns in the DataFrame df_merged related to tax revenue loss using different methods (CIT and ETR) and percentage totals, based on given tax base gain and loss values
for st in ["","Min. ","Max. "]:
    #Total tax lost
    #OW already in the file
    df_merged[f"{st}Base gain TA (USD million)"] = df_merged[f'TA: {st}Tax base gain (USD million)'].fillna(0)
    df_merged[f"{st}Harm TA: Total (% total)"] = 100*df_merged[f"{st}Base gain TA (USD million)"]/df_merged[f"{st}Base gain TA (USD million)"].sum()
    
    total_loss_ta_cit = df_merged[f"{st}Loss TA using CIT (USD million)"].sum()
    total_loss_ta_etr = df_merged[f"{st}Loss TA using ETR (USD million)"].sum()
    df_merged[f"{st}Harm TA: Total using CIT (USD million)"] = total_loss_ta_cit*df_merged[f"{st}Harm TA: Total (% total)"].fillna(0)/100
    df_merged[f"{st}Harm TA: Total using ETR (USD million)"] = total_loss_ta_etr*df_merged[f"{st}Harm TA: Total (% total)"].fillna(0)/100

# Calculate various columns in the DataFrame df_merged related to total harm using different tax methods (CIT and ETR), including total harm values in USD million and percentage of the total, as well as the number of nurses affected based on the total harm percentage.
for tax in ["CIT","ETR"]:
    total_loss = df_merged[f"Loss Total using {tax} (USD million)"].sum()
    df_merged[f"Harm: Total using {tax} (USD million)"] = df_merged[f"Harm TA: Total using {tax} (USD million)"] + df_merged["Harm OW: Total (USD million)"]
    df_merged[f"Harm: Total using {tax} (% total)"] = 100*df_merged[f"Harm: Total using {tax} (USD million)"]/(df_merged[f"Harm TA: Total using {tax} (USD million)"] + df_merged["Harm OW: Total (USD million)"]).sum()

    total_nurses = df_merged[f'Loss Total using {tax} (# Nurses)'].sum()
    df_merged[f"Harm: Total using {tax} (# Nurses)"] = total_nurses*df_merged[f"Harm: Total using {tax} (% total)"]/100

df_merged["Year"] = 2021

#Add missing values
# Set the values of specific columns in the DataFrame df_merged (including "OW: Tax base loss (USD million)", "OW: Tax revenue loss (USD million)", "Harm OW: Total (% total)", "Harm OW: Total (USD million)", and "Loss OW (USD million)") to NaN if the corresponding values in the column "OW: Tax base loss (USD million)" are NaN.
df_merged.loc[np.isnan(df_merged["OW: Tax base loss (USD million)"]),
              ['OW: Tax base loss (USD million)',
 'OW: Tax revenue loss (USD million)',
 'Harm OW: Total (% total)',
 'Harm OW: Total (USD million)','Loss OW (USD million)']] = np.NaN

# Set specific columns in the DataFrame df_merged to NaN if both "TA: Tax base loss (USD million)" and "OW: Tax base loss (USD million)" have NaN values
cond = np.isnan(df_merged["TA: Tax base loss (USD million)"]) & np.isnan(df_merged["OW: Tax base loss (USD million)"])
df_merged.loc[cond,['Loss Total (USD million)',
 'Loss Total (% GDP)',
 'Loss Total (% gvt tax revenue)',
 'Loss Total Global (% GDP)',
 'Loss Total Regional (% GDP)',
 'Loss Total Global (% gvt tax revenue)',
 'Loss Total Regional (% gvt tax revenue)',
 'Loss Total (per capita)',
 'Loss Total (% Education)',
 'Loss Total (% Health)',
 'Loss Total (%  gvt corporate revenue)',
 'Loss Total (%  gvt tax revenue)',
 'Loss Total (# Nurses)']] = np.nan

In [57]:
# Perform an outer merge of the DataFrame df_merged with another DataFrame named other_info
df_merged = pd.merge(df_merged,other_info,how="outer")
df_merged.loc[df_merged["Country"]=="India"]

,Share of global offshore wealth owned by citizens of country,OW: Tax base loss (USD million),Offshore wealth owned by citizens of country (% of GDP),OW: Tax revenue loss (USD million),Harm OW: Total (% total),Harm OW: Total (USD million),ISO-3 code of country,Profit loss (M) - dom,Min. Profit loss (M) - dom,Max. Profit loss (M) - dom,TA: Reporting country,ETR,Profit loss (M) - for lose,Min. Profit loss (M) - for lose,Max. Profit loss (M) - for lose,Revenue loss using CIT (M) - for lose,Revenue loss using ETR (M) - for lose,Robust - for lose,TA: Number countries reporting,CIT,Profit loss (M) - for gain,Min. Profit loss (M) - for gain,Max. Profit loss (M) - for gain,Revenue loss using CIT (M) - for gain,Revenue loss using ETR (M) - for gain,Robust - for gain,Revenue loss using CIT (M) - dom,Revenue loss using ETR (M) - dom,TA: Tax base gain (USD million),TA: Tax base loss (USD million),TA: Min. Tax base gain (USD million),TA: Min. Tax base loss (USD million),TA: Max. Tax base gain (USD million),TA: Max. Tax base loss (USD million),TA: Tax revenue gain using CIT (USD million),TA: Tax revenue gain using ETR (USD million),TA: Tax revenue loss using CIT (USD million),TA: Tax revenue loss using ETR (USD million),TA: Min. Tax revenue gain using CIT (USD million),TA: Min. Tax revenue gain using ETR (USD million),TA: Min. Tax revenue loss using CIT (USD million),TA: Min. Tax revenue loss using ETR (USD million),TA: Max. Tax revenue gain using CIT (USD million),TA: Max. Tax revenue gain using ETR (USD million),TA: Max. Tax revenue loss using CIT (USD million),TA: Max. Tax revenue loss using ETR (USD million),TA: Robust,iso3,GRD: Total tax revenue,GRD: Total corporate income revenue,...,Loss Total using ETR (% gvt corporate revenue),Loss Total using ETR (% gvt tax revenue),Loss Total using ETR (# Nurses),Base gain TA (USD million),Harm TA: Total (% total),Harm TA: Total using CIT (USD million),Harm TA: Total using ETR (USD million),Min. Base gain TA (USD million),Min. Harm TA: Total (% total),Min. Harm TA: Total using CIT (USD million),Min. Harm TA: Total using ETR (USD million),Max. Base gain TA (USD million),Max. Harm TA: Total (% total),Max. Harm TA: Total using CIT (USD million),Max. Harm TA: Total using ETR (USD million),Harm: Total using CIT (USD million),Harm: Total using CIT (% total),Harm: Total using CIT (# Nurses),Harm: Total using ETR (USD million),Harm: Total using ETR (% total),Harm: Total using ETR (# Nurses),Year,Loss Total (USD million),Loss Total (% GDP),Loss Total (% gvt tax revenue),Loss Total Global (% GDP),Loss Total Regional (% GDP),Loss Total Global (% gvt tax revenue),Loss Total Regional (% gvt tax revenue),Loss Total (per capita),Loss Total (% Education),Loss Total (% Health),Loss Total (% gvt corporate revenue),Loss Total (% gvt tax revenue),Loss Total (# Nurses),total_taxes_revenue,cit_revenue,govt_exp_educ_gdp,population,month_wage,who_gvt_health_expenditure,gdp,cthi_2021_rank,cthi_2021_share,cthi_2021_score,fsi_2022_rank,fsi_2022_score,region_tjn,ukt,income_class
155,0.001017,505.768108,0.003524,181.469604,0.0,0.0,IND,33781.0,33541.0,34087.0,1.0,0.413186,29565.0,28204.0,31243.0,15096.6176,12909.6076,2.0,22.0,0.48316,0.0,0.0,0.0,0.0,0.0,1.0,16470.8384,14084.7484,-0.0,63346.0,-0.0,61745.0,-0.0,65330.0,-0.0,-0.0,30606.254559,26173.661596,-0.0,-0.0,29832.715368,25512.151284,-0.0,-0.0,31564.844036,26993.422032,2.0,IND,NaN,NaN,...,NaN,NaN,9.347780e+06,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,0.0,0.0,0.0,2021.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.179488e+11,1.369003e+09,234.95,2.582103e+10,2.702930e+12,NaN,NaN,NaN,36.0,54.725,Asia,0.0,Lower-middle income


In [58]:
def robust(s):
    return a

# Iterate over each row in the DataFrame df_merged and assigns a background color based on the value of the "TA: Robust" column, storing the colors in a list called a.
a = []
for i,row in df_merged.iterrows():
    if row["TA: Robust"]==2:
        a.append("background-color: #44b0c6")
    elif row["TA: Robust"]==1:
        a.append("background-color: #94c9d4")
    else:
        a.append("background-color: white")
        
    
# df_merged.style.apply(robust)

In [59]:
# Applies the "robust" style to the DataFrame df_merged and saves it as an Excel file in different locations specified by the file paths provided.
df_merged.style.apply(robust).to_excel("~/Downloads/combined_output.xlsx",index=None)
df_merged.style.apply(robust).to_excel(final_table_output,index=None)
df_merged.style.apply(robust).to_excel(final_table_output_workstream,index=None)

In [ ]:
# Select the rows in the DataFrame df_merged where the value in the "Income Class" column, after filling missing values with "X", is equal to "X".
df_merged.loc[df_merged["Income Class"].fillna("X")=="X"]

In [ ]:
# ## Work for ATP (danish people)
# #Main results
# final_data_output = f"{path_files_temp}{YEAR_CBCR}_replicates.csv"
# atp_prbook = pd.read_csv(final_data_output,sep="\t").reset_index(drop=True)
# atp_prbook = atp_prbook.dropna()
# atp_prbook = atp_prbook.loc[atp_prbook["profits"]>0]
# atp_prbook = atp_prbook.groupby(["iso3_d","n_rep"]).sum()[["profits"]].groupby("iso3_d").median().reset_index()
# atp_prbook.head()



In [ ]:
# atp_other = pd.read_excel(final_table_output_workstream)
# atp_other["Gain - Loss"] = atp_other["TA: Tax base gain (USD million)"] -  atp_other["TA: Tax base loss (USD million)"]
# atp_other = atp_other[['ISO-3 code of country','ETR', "Gain - Loss"]]
# atp_other.columns = ["iso3_d","ETR","Gain - Loss"]
# atp_other = atp_other.loc[atp_other["Gain - Loss"]>0]
# atp_other = pd.merge(atp_other,atp_prbook,how="left")
# atp_other["profits"] /= 1E6

In [ ]:
# atp_other["var"] = 100*atp_other["Gain - Loss"]/atp_other["profits"] * atp_other["Gain - Loss"]/atp_other["Gain - Loss"].sum()
# atp_other=  atp_other.sort_values(by="var",ascending=False)
# atp_other.to_excel("C:/Users/javga/Downloads/temp.xlsx")

In [ ]:
# atp_other.loc[atp_other["profits"]<atp_other["Gain - Loss"]]